# Hyperparameter Experiments: DeBERTa-v3-base
##### Ömer Faruk Merey - Middle East Technical University

Systematic hyperparameter search for `microsoft/deberta-v3-base` on the BrownBox customer service sentiment dataset.

**Experiments** (each changes ONE variable from baseline):

| Group | Parameter | Values Tested |
|-------|-----------|---------------|
| Baseline | — | LR=2e-5, epochs=5, batch=8, dropout=0.1, CE+balanced weights, head/tail=254/255 |
| Learning Rate | LR | 1e-5, 5e-5 |
| Epochs | epochs | 3, 7, 10 |
| Batch Size | batch_size | 4, 16 |
| Loss Function | loss | Focal Loss (γ=2), CE without weights, CE with boosted positive (40x) |
| Regularization | dropout / decay / freeze | dropout=0.2, dropout=0.3, weight_decay=0.1, freeze bottom 6 layers |
| Truncation | head/tail split | tail-heavy (128/381), head-heavy (381/128) |

All results logged to **wandb** project: `customer-sentiment-analysis`

## 1. Setup & Load Data

In [12]:
import pandas as pd
import numpy as np
import json
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, accuracy_score
import wandb
import gc

# Kill any stale wandb runs from previous cells
try:
    wandb.finish(quiet=True)
except:
    pass

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


In [13]:
%cd /content/drive/MyDrive/attention-based-customer-sentiment-analysis/train

/content/drive/MyDrive/attention-based-customer-sentiment-analysis/train


In [14]:
# Load preprocessed data
train_df = pd.read_csv('../dataset/processed/train_split.csv')
val_df = pd.read_csv('../dataset/processed/val_split.csv')

with open('../dataset/processed/metadata.json') as f:
    metadata = json.load(f)

label_map = metadata['label_map']
label_map_inv = {v: k for k, v in label_map.items()}
NUM_LABELS = len(label_map)

print(f"Train: {len(train_df)}, Val: {len(val_df)}")
print(f"Labels: {label_map}")
print(f"Class weights: {[round(w, 3) for w in metadata['class_weights']]}")

Train: 776, Val: 194
Labels: {'negative': 0, 'neutral': 1, 'positive': 2}
Class weights: [0.786, 0.596, 19.897]


## 2. Helper Functions

In [15]:
MODEL_NAME = 'microsoft/deberta-v3-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def head_tail_tokenize(texts, tokenizer, max_length=512, head_ratio=0.5):
    """Tokenize with configurable head/tail split ratio."""
    all_input_ids = []
    all_attention_masks = []

    usable_tokens = max_length - 3  # [CLS] ... [SEP] ... [SEP]
    head_len = int(usable_tokens * head_ratio)
    tail_len = usable_tokens - head_len

    for text in texts:
        tokens = tokenizer.encode(text, add_special_tokens=False)

        if len(tokens) <= max_length - 2:
            encoding = tokenizer(
                text, max_length=max_length, padding='max_length',
                truncation=True, return_tensors='pt'
            )
        else:
            head = tokens[:head_len]
            tail = tokens[-tail_len:]
            combined = [tokenizer.cls_token_id] + head + [tokenizer.sep_token_id] + tail + [tokenizer.sep_token_id]
            assert len(combined) == max_length, f"Expected {max_length}, got {len(combined)}"
            encoding = {
                'input_ids': torch.tensor([combined]),
                'attention_mask': torch.tensor([[1] * max_length])
            }

        all_input_ids.append(encoding['input_ids'].squeeze(0))
        all_attention_masks.append(encoding['attention_mask'].squeeze(0))

    return {
        'input_ids': torch.stack(all_input_ids),
        'attention_mask': torch.stack(all_attention_masks)
    }

class SentimentDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = torch.tensor(labels.values, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels': self.labels[idx]
        }

class FocalLoss(nn.Module):
    """Focal Loss for addressing class imbalance — down-weights easy examples."""
    def __init__(self, weight=None, gamma=2.0):
        super().__init__()
        self.weight = weight
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, weight=self.weight, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

print("Helpers defined: head_tail_tokenize, SentimentDataset, FocalLoss")

Helpers defined: head_tail_tokenize, SentimentDataset, FocalLoss


In [16]:
# Pre-tokenize for default (50/50) split — reused by most experiments
print("Tokenizing with default head/tail split (50/50)...")
enc_default_train = head_tail_tokenize(train_df['conversation'].tolist(), tokenizer, head_ratio=0.5)
enc_default_val = head_tail_tokenize(val_df['conversation'].tolist(), tokenizer, head_ratio=0.5)

# Pre-tokenize for tail-heavy split (25/75)
print("Tokenizing with tail-heavy split (25/75)...")
enc_tail_train = head_tail_tokenize(train_df['conversation'].tolist(), tokenizer, head_ratio=0.25)
enc_tail_val = head_tail_tokenize(val_df['conversation'].tolist(), tokenizer, head_ratio=0.25)

# Pre-tokenize for head-heavy split (75/25)
print("Tokenizing with head-heavy split (75/25)...")
enc_head_train = head_tail_tokenize(train_df['conversation'].tolist(), tokenizer, head_ratio=0.75)
enc_head_val = head_tail_tokenize(val_df['conversation'].tolist(), tokenizer, head_ratio=0.75)

print("Done. 3 tokenization variants ready.")

Tokenizing with default head/tail split (50/50)...
Tokenizing with tail-heavy split (25/75)...
Tokenizing with head-heavy split (75/25)...
Done. 3 tokenization variants ready.


## 3. Experiment Configurations

In [17]:
# Baseline config — each experiment overrides ONE parameter
BASELINE = {
    'lr': 2e-5,
    'epochs': 5,
    'batch_size': 8,
    'warmup_ratio': 0.1,
    'weight_decay': 0.01,
    'grad_clip': 1.0,
    'dropout': 0.1,
    'loss': 'ce_balanced',       # ce_balanced | ce_none | ce_boosted | focal
    'freeze_layers': 0,          # number of encoder layers to freeze from bottom
    'encodings': 'default',      # default | tail_heavy | head_heavy
}

def make_exp(name, **overrides):
    """Create experiment config from baseline with overrides."""
    cfg = BASELINE.copy()
    cfg['name'] = name
    cfg.update(overrides)
    return cfg

EXPERIMENTS = [
    # --- Baseline ---
    make_exp('baseline'),

    # --- Learning Rate ---
    make_exp('lr-1e5', lr=1e-5),
    make_exp('lr-5e5', lr=5e-5),

    # --- Epochs ---
    make_exp('epochs-3', epochs=3),
    make_exp('epochs-7', epochs=7),
    make_exp('epochs-10', epochs=10),

    # --- Batch Size ---
    make_exp('batch-4', batch_size=4),
    make_exp('batch-16', batch_size=16),

    # --- Loss Function ---
    make_exp('focal-loss', loss='focal'),
    make_exp('ce-no-weights', loss='ce_none'),
    make_exp('ce-boosted-pos', loss='ce_boosted'),

    # --- Regularization ---
    make_exp('dropout-0.2', dropout=0.2),
    make_exp('dropout-0.3', dropout=0.3),
    make_exp('weight-decay-0.1', weight_decay=0.1),
    make_exp('freeze-6-layers', freeze_layers=6),

    # --- Truncation Strategy ---
    make_exp('tail-heavy-25-75', encodings='tail_heavy'),
    make_exp('head-heavy-75-25', encodings='head_heavy'),
]

print(f"Total experiments: {len(EXPERIMENTS)}")
for i, exp in enumerate(EXPERIMENTS):
    diff = {k: v for k, v in exp.items() if k != 'name' and v != BASELINE.get(k)}
    label = ', '.join(f'{k}={v}' for k, v in diff.items()) if diff else 'baseline'
    print(f"  {i+1:2d}. {exp['name']:25s} [{label}]")

Total experiments: 17
   1. baseline                  [baseline]
   2. lr-1e5                    [lr=1e-05]
   3. lr-5e5                    [lr=5e-05]
   4. epochs-3                  [epochs=3]
   5. epochs-7                  [epochs=7]
   6. epochs-10                 [epochs=10]
   7. batch-4                   [batch_size=4]
   8. batch-16                  [batch_size=16]
   9. focal-loss                [loss=focal]
  10. ce-no-weights             [loss=ce_none]
  11. ce-boosted-pos            [loss=ce_boosted]
  12. dropout-0.2               [dropout=0.2]
  13. dropout-0.3               [dropout=0.3]
  14. weight-decay-0.1          [weight_decay=0.1]
  15. freeze-6-layers           [freeze_layers=6]
  16. tail-heavy-25-75          [encodings=tail_heavy]
  17. head-heavy-75-25          [encodings=head_heavy]


## 4. Training Engine

In [18]:
def get_encodings(cfg):
    """Return pre-tokenized encodings based on config."""
    if cfg['encodings'] == 'tail_heavy':
        return enc_tail_train, enc_tail_val
    elif cfg['encodings'] == 'head_heavy':
        return enc_head_train, enc_head_val
    return enc_default_train, enc_default_val

def get_criterion(cfg, device):
    """Build loss function based on config."""
    balanced_weights = torch.tensor(metadata['class_weights'], dtype=torch.float32).to(device)

    if cfg['loss'] == 'focal':
        return FocalLoss(weight=balanced_weights, gamma=2.0)
    elif cfg['loss'] == 'ce_none':
        return nn.CrossEntropyLoss()
    elif cfg['loss'] == 'ce_boosted':
        boosted = balanced_weights.clone()
        boosted[2] = 40.0
        return nn.CrossEntropyLoss(weight=boosted)
    else:  # ce_balanced
        return nn.CrossEntropyLoss(weight=balanced_weights)

def build_model(cfg, device):
    """Build model with optional dropout override."""
    config = AutoConfig.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS,
        hidden_dropout_prob=cfg['dropout'],
        attention_probs_dropout_prob=cfg['dropout'],
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, config=config, torch_dtype=torch.float32
    )

    # Freeze bottom N encoder layers if requested
    if cfg['freeze_layers'] > 0:
        for i in range(cfg['freeze_layers']):
            for param in model.deberta.encoder.layer[i].parameters():
                param.requires_grad = False
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        total = sum(p.numel() for p in model.parameters())
        print(f"  Frozen {cfg['freeze_layers']} layers — trainable: {trainable:,}/{total:,}")

    model.to(device)
    return model

def evaluate(model, loader, criterion, device):
    """Evaluate model, return loss, preds, labels."""
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits.float()  # ensure float32 for loss
            loss = criterion(logits, labels)
            total_loss += loss.item()

            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return total_loss / len(loader), np.array(all_preds), np.array(all_labels)

print("Training engine ready.")

Training engine ready.


In [19]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [20]:
def run_experiment(cfg):
    """Run a single experiment: build model, train, evaluate, log to wandb."""
    exp_name = cfg['name']
    print(f"\n{'='*60}")
    print(f"  EXPERIMENT: {exp_name}")
    print(f"{'='*60}")

    # Init wandb
    wandb.init(
        project='customer-sentiment-analysis',
        name=f'exp-{exp_name}',
        config=cfg,
        reinit=True
    )

    # Data
    train_enc, val_enc = get_encodings(cfg)
    train_dataset = SentimentDataset(train_enc, train_df['label'])
    val_dataset = SentimentDataset(val_enc, val_df['label'])
    train_loader = DataLoader(train_dataset, batch_size=cfg['batch_size'], shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=cfg['batch_size'])

    # Model
    model = build_model(cfg, device)
    criterion = get_criterion(cfg, device)

    # Only optimize unfrozen params
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable_params, lr=cfg['lr'], weight_decay=cfg['weight_decay'])

    total_steps = len(train_loader) * cfg['epochs']
    warmup_steps = int(total_steps * cfg['warmup_ratio'])
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    # Training loop
    best_val_f1 = 0
    best_epoch = 0

    for epoch in range(cfg['epochs']):
        model.train()
        total_train_loss = 0

        for step, batch in enumerate(train_loader):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs.logits.float(), labels)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=cfg['grad_clip'])
            optimizer.step()
            scheduler.step()
            total_train_loss += loss.item()

        avg_train_loss = total_train_loss / len(train_loader)

        # Validation
        val_loss, val_preds, val_labels = evaluate(model, val_loader, criterion, device)
        val_acc = accuracy_score(val_labels, val_preds)
        val_f1_w = f1_score(val_labels, val_preds, average='weighted')
        val_f1_m = f1_score(val_labels, val_preds, average='macro')
        val_f1_per = f1_score(val_labels, val_preds, average=None, zero_division=0)

        wandb.log({
            'epoch': epoch + 1,
            'train/loss': avg_train_loss,
            'val/loss': val_loss,
            'val/accuracy': val_acc,
            'val/f1_weighted': val_f1_w,
            'val/f1_macro': val_f1_m,
            'val/f1_negative': val_f1_per[0],
            'val/f1_neutral': val_f1_per[1],
            'val/f1_positive': val_f1_per[2],
        })

        marker = ""
        if val_f1_w > best_val_f1:
            best_val_f1 = val_f1_w
            best_epoch = epoch + 1
            marker = " *"

        print(f"  Epoch {epoch+1}/{cfg['epochs']} | "
              f"TrLoss: {avg_train_loss:.4f} | "
              f"VlLoss: {val_loss:.4f} | "
              f"Acc: {val_acc:.4f} | "
              f"F1w: {val_f1_w:.4f} | "
              f"F1m: {val_f1_m:.4f}{marker}")

    # Log best results as summary
    wandb.summary['best_val_f1_weighted'] = best_val_f1
    wandb.summary['best_epoch'] = best_epoch
    wandb.finish()

    # Cleanup
    del model, optimizer, scheduler, criterion
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {'name': exp_name, 'best_val_f1_w': best_val_f1, 'best_epoch': best_epoch}

print("run_experiment() ready.")

run_experiment() ready.


## 5. Run All Experiments

In [21]:
results = []
for i, cfg in enumerate(EXPERIMENTS):
    print(f"\n[{i+1}/{len(EXPERIMENTS)}]", end="")
    result = run_experiment(cfg)
    results.append(result)
    print(f"  >> Best F1(w): {result['best_val_f1_w']:.4f} at epoch {result['best_epoch']}")

print(f"\n{'='*60}")
print(f"  ALL {len(EXPERIMENTS)} EXPERIMENTS COMPLETE")
print(f"{'='*60}")


[1/17]
  EXPERIMENT: baseline


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


...

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.weight        

  Epoch 1/5 | TrLoss: 0.9635 | VlLoss: 1.0704 | Acc: 0.6598 | F1w: 0.6509 | F1m: 0.4385 *
  Epoch 2/5 | TrLoss: 0.9840 | VlLoss: 1.0326 | Acc: 0.6598 | F1w: 0.6509 | F1m: 0.4385
  Epoch 3/5 | TrLoss: 0.9280 | VlLoss: 0.9519 | Acc: 0.8351 | F1w: 0.8265 | F1m: 0.5617 *
  Epoch 4/5 | TrLoss: 0.7582 | VlLoss: 0.9534 | Acc: 0.8093 | F1w: 0.8011 | F1m: 0.5446
  Epoch 5/5 | TrLoss: 0.6073 | VlLoss: 0.8880 | Acc: 0.8402 | F1w: 0.8316 | F1m: 0.5654 *


epoch,▁▃▅▆█
train/loss,██▇▄▁
val/accuracy,▁▁█▇█
val/f1_macro,▁▁█▇█
val/f1_negative,▁▁█▇█
val/f1_neutral,▁▁█▇█
val/f1_positive,▁▁▁▁▁
val/f1_weighted,▁▁█▇█
val/loss,█▇▃▄▁
best_epoch,5
best_val_f1_weighted,0.83158


  >> Best F1(w): 0.8316 at epoch 5

  ALL 17 EXPERIMENTS COMPLETE


## 6. Results Summary

In [22]:
# Leaderboard — sorted by best validation F1 (weighted)
results_df = pd.DataFrame(results).sort_values('best_val_f1_w', ascending=False).reset_index(drop=True)
results_df.index += 1
results_df.columns = ['Experiment', 'Best Val F1 (weighted)', 'Best Epoch']
results_df['Best Val F1 (weighted)'] = results_df['Best Val F1 (weighted)'].round(4)
print("Experiment Leaderboard:")
results_df

Experiment Leaderboard:


,Experiment,Best Val F1 (weighted),Best Epoch
1,epochs-10,0.8874,9
2,focal-loss,0.8722,5
3,batch-4,0.8672,4
4,ce-no-weights,0.8570,5
5,tail-heavy-25-75,0.8469,5
6,epochs-7,0.8414,7
7,baseline,0.8367,5
8,head-heavy-75-25,0.8316,5
9,ce-boosted-pos,0.8263,5
10,weight-decay-0.1,0.8049,5
